In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from typing import List, Tuple


In [3]:

# Set random seed for reproducibility
np.random.seed(42)
random.seed(42)

def generate_hard_set(n_samples: int = 10000) -> pd.DataFrame:
    """Generate a challenging dataset for rent classification"""
    
    # 1. SYNONYMS & IMPLIED RENT (No direct keywords)
    rent_synonyms = [
        # Formal/legal terms
        "Lease obligation payment",
        "Tenancy remittance",
        "Residential occupancy fee",
        "Domicile disbursement",
        "Habitation remittance",
        "Premises maintenance remittance",
        "Abode disbursement",
        "Dwelling remittance",
        "Residence disbursement",
        # Corporate/institutional terms
        "Recurring habitation expenditure",
        "Periodic domicile remittance",
        "Monthly premises fee",
        "Housing allocation payment",
        "Residential facility fee",
        "Accommodation disbursement",
        "Lodging remittance",
        # Vague but contextually clear
        "Where I live payment",
        "Place I stay remittance",
        "My apartment payment",
        "My house monthly",
        "My unit disbursement",
        # Property management terms
        "Property management remittance",
        "Real estate management fee",
        "Landlord services payment",
        "Building administration fee",
        "Complex management remittance",
        "Facility management disbursement",
        # Invoice/reference based
        "Invoice #APT2024-001",
        "Ref: RES-UNIT-5B",
        "Payment ref: HOUSING-",
        "Invoice: RESIDENCE",
        "Remittance: DWELLING",
        # Address-based (no explicit rent)
        "123 Main St monthly",
        "500 Oak Ave payment",
        "For 303 Parkview",
        "Unit 7B monthly",
        "Apartment 12C fee"
    ]
    
    # 2. AMBIGUOUS TRANSACTIONS (Could be rent OR something else)
    ambiguous_phrases = [
        # Without context, could be anything
        "Monthly payment to John",
        "Recurring transfer to Smith",
        "ACH to Johnson LLC",
        "Check #1234 for monthly",
        "Debit to Anderson",
        "Wire to Thompson",
        "Zelle to Davis",
        "Transfer to Miller",
        "Payment to Wilson",
        "Remittance to Moore",
        # Property-related but ambiguous
        "Property Services LLC",
        "Real Estate Solutions",
        "Housing Associates",
        "Residential Services",
        "Building Management",
        "Facility Solutions",
        "Complex Services",
        "Unit Management",
        "Premises Services",
        "Domicile Solutions",
        # Amount/date patterns only
        "Payment on 1st of month",
        "Monthly on 15th",
        "Recurring 1st week",
        "First of month payment",
        "Mid-month transfer",
        "End of month remittance",
        "Periodic 30th payment",
        "Bi-weekly housing",
        "Semi-monthly residence",
        "Quarterly dwelling"
    ]
    
    # 3. HOMONYMS & POLYSEMY (Same word, different meanings)
    homonym_contexts = [
        # "Unit" could be apartment OR measurement/business unit
        "Unit payment",
        "Unit fee",
        "Unit charge",
        "Unit disbursement",
        "Unit remittance",
        # "Complex" could be apartment complex OR complicated
        "Complex fee",
        "Complex payment",
        "Complex charge",
        # "Building" could be rent OR construction
        "Building payment",
        "Building fee",
        "Building disbursement",
        # "Property" could be rent OR insurance/tax
        "Property payment",
        "Property fee",
        "Property remittance",
        # "Lease" could be apartment OR car/equipment
        "Lease payment",
        "Lease fee",
        "Lease charge"
    ]
    
    # 4. NEGATION & EXCEPTIONS
    negations = [
        "NOT rent - utilities",
        "Non-rent housing expense",
        "Excluding rent - HOA",
        "Besides rent - insurance",
        "Other than rent - tax",
        "Additional to rent - fee",
        "Separate from rent - deposit",
        "Independent of rent - repair",
        "Distinct from rent - service",
        "Apart from rent - maintenance"
    ]
    
    # 5. CULTURAL/LINGUISTIC VARIATIONS
    cultural_terms = [
        # British English
        "Letting fee",
        "Tenancy payment",
        "Lodgings remittance",
        "Digs payment",
        # Australian
        "Rental payment",
        "Flat fee",
        "Unit letting",
        # Canadian
        "Condo fee",
        "Strata payment",
        # Informal/slang
        "Pad payment",
        "Crib fee",
        "Spot payment",
        "Place remittance"
    ]
    
    # 6. COMPOUND/COMPLEX DESCRIPTIONS
    compounds = [
        "Monthly housing and utilities bundle",
        "Residence fee including amenities",
        "Apartment payment + parking",
        "Rent with included services",
        "Dwelling disbursement plus fees",
        "Habitation remittance with utilities",
        "Unit payment including maintenance",
        "Premises fee with service charge",
        "Abode payment plus insurance",
        "Accommodation remittance including tax"
    ]
    
    # 7. MISSPELLINGS & TYPOS
    misspellings = [
        "Rnet payment",
        "Rant fee",
        "Rentt disbursement",
        "Rant remittance",
        "Rent (missed auto)",
        "Ren tpayment",
        "R3nt fee",
        "Rent-paymnt",
        "Rent-pay ment",
        "Rent(payment)",
        "Rent..payment",
        "Rent ;payment",
        "Rent/payment"
    ]
    
    # 8. FORMAT VARIATIONS
    formats = [
        "RENT - APT 5B",
        "RENT: UNIT 304",
        "RENT PAYMENT (MONTHLY)",
        "RENT - DO NOT DELETE",
        "RENT #2024-001",
        "RENT_INVOICE_001",
        "RENT-RECURRING",
        "RENT~MONTHLY",
        "RENT|AUTO",
        "RENT>>PAYMENT"
    ]
    
    # 9. CONTEXT-DEPENDENT AMOUNTS
    # Rent-like amounts but could be other things
    rent_amounts = [650, 750, 850, 950, 1100, 1250, 1350, 1450, 1550, 1650, 1750, 1850, 1950, 2100, 2250, 2400, 2600, 2800, 3000]
    non_rent_amounts = [50, 75, 100, 125, 150, 200, 250, 300, 350, 400, 450, 500, 600, 700, 800, 900, 1000, 1200, 1400, 1600, 1800]
    
    # 10. TEMPORAL PATTERNS
    date_patterns = [
        "1st of month", "5th monthly", "10th recurring", "15th auto", "20th periodic",
        "25th standing", "Last day", "First Monday", "Bi-weekly", "Semi-monthly",
        "Quarterly", "Every 30 days", "Monthly anniversary", "Calendar month"
    ]
    
    # Generate the dataset
    data = []
    
    for i in range(n_samples):
        # Randomly decide if this is rent (balanced dataset)
        is_rent = random.choice([True, False])
        
        # Generate date (spanning 2 years)
        start_date = datetime(2024, 1, 1)
        days_diff = random.randint(0, 730)
        current_date = start_date + timedelta(days=days_diff)
        
        # Select description category based on difficulty level
        difficulty = random.choices(
            ['synonym', 'ambiguous', 'homonym', 'negation', 'cultural', 
             'compound', 'misspelling', 'format', 'temporal'],
            weights=[0.15, 0.15, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]
        )[0]
        
        description = ""
        
        if difficulty == 'synonym':
            description = random.choice(rent_synonyms) if is_rent else random.choice(ambiguous_phrases)
        elif difficulty == 'ambiguous':
            # Use ambiguous phrases for both rent and non-rent
            description = random.choice(ambiguous_phrases)
            # Add subtle contextual clues for rent
            if is_rent and random.random() < 0.3:
                description += f" (Unit {random.randint(1, 50)}{random.choice(['A', 'B', 'C'])})"
        elif difficulty == 'homonym':
            description = random.choice(homonym_contexts)
            # Rent clues in amount/date
            if is_rent:
                description += f" for residential unit"
        elif difficulty == 'negation':
            if is_rent:
                description = f"Rent payment {random.choice(['', '(primary residence)', '(apartment)'])}"
            else:
                description = random.choice(negations)
        elif difficulty == 'cultural':
            description = random.choice(cultural_terms)
        elif difficulty == 'compound':
            description = random.choice(compounds) if is_rent else f"{random.choice(['Utility', 'Service', 'Maintenance', 'HOA'])} fee {random.choice(['+ tax', '+ insurance', 'incl. services'])}"
        elif difficulty == 'misspelling':
            if is_rent:
                description = random.choice(misspellings)
            else:
                description = f"{random.choice(['Utilitie', 'Servise', 'Maintenence', 'Subscripton'])} payment"
        elif difficulty == 'format':
            description = random.choice(formats) if is_rent else f"{random.choice(['UTILITY', 'SERVICE', 'FEE', 'CHARGE'])} - {random.choice(['MONTHLY', 'RECURRING', 'AUTO'])}"
        elif difficulty == 'temporal':
            temporal = random.choice(date_patterns)
            if is_rent:
                description = f"Residential payment {temporal}"
            else:
                description = f"{random.choice(['Service', 'Subscription', 'Membership'])} {temporal}"
        
        # Add amount noise
        if is_rent:
            base_amount = random.choice(rent_amounts)
            # Add small variations
            amount = base_amount + random.choice([-50, -25, 0, 25, 50])
        else:
            amount = random.choice(non_rent_amounts)
            # Sometimes use rent-like amounts for non-rent (challenge!)
            if random.random() < 0.2:
                amount = random.choice(rent_amounts)
        
        # Format amount (negative for payments)
        formatted_amount = f"-${amount:,.2f}" if random.random() < 0.9 else f"${amount:,.2f}"
        
        # Generate realistic balance
        balance = random.randint(1000, 50000)
        
        # Create transaction record
        record = {
            'Date': current_date.strftime('%m/%d/%Y'),
            'Description': description,
            'Comments': '',  # Empty for simplicity
            'Check Number': '',  # Empty for simplicity
            'Amount': formatted_amount,
            'Balance': f"${balance:,.2f}",
            'rent': 1 if is_rent else 0
        }
        
        data.append(record)
    
    return pd.DataFrame(data)


In [4]:

# Generate 10,000 hard examples
df_hard = generate_hard_set(10000)

# Save to CSV
df_hard.to_csv('hard_rent_classification_10000.csv', index=False)

# Create a smaller sample for display
sample_df = df_hard.sample(20, random_state=42)

print("Generated 10,000 hard examples!")
print("\nSample of 20 transactions (10 rent, 10 non-rent):")
print("="*80)
for idx, row in sample_df.iterrows():
    label = "RENT" if row['rent'] == 1 else "NOT RENT"
    print(f"{row['Date']} | {row['Description']:40} | {row['Amount']:>10} | {label}")
print("="*80)

# Calculate statistics
rent_count = df_hard['rent'].sum()
non_rent_count = len(df_hard) - rent_count

print(f"\nDataset Statistics:")
print(f"Total transactions: {len(df_hard):,}")
print(f"Rent transactions: {rent_count:,} ({rent_count/len(df_hard)*100:.1f}%)")
print(f"Non-rent transactions: {non_rent_count:,} ({non_rent_count/len(df_hard)*100:.1f}%)")

# Show difficulty distribution
print("\nChallenge Categories in Dataset:")
challenge_types = [
    "Synonyms & implied rent",
    "Ambiguous patterns",
    "Homonyms & polysemy",
    "Negation & exceptions",
    "Cultural variations",
    "Compound descriptions",
    "Misspellings & typos",
    "Format variations",
    "Temporal patterns"
]
for i, category in enumerate(challenge_types):
    print(f"  {category}: ~{10000//len(challenge_types):,} examples")

Generated 10,000 hard examples!

Sample of 20 transactions (10 rent, 10 non-rent):
11/26/2024 | Unit disbursement                        |   -$600.00 | NOT RENT
03/07/2025 | Servise payment                          | -$2,800.00 | NOT RENT
11/15/2024 | Residential payment 5th monthly          | -$1,825.00 | RENT
11/01/2024 | Spot payment                             | -$1,900.00 | RENT
12/22/2025 | Service 10th recurring                   |   -$300.00 | NOT RENT
07/31/2024 | Flat fee                                 | -$1,225.00 | RENT
07/17/2025 | Utility fee + tax                        |   -$600.00 | NOT RENT
09/19/2025 | Membership 25th standing                 |  $1,950.00 | NOT RENT
12/09/2025 | Rent payment                             | -$1,575.00 | RENT
02/13/2024 | Rent payment (apartment)                 |  $2,075.00 | RENT
05/23/2025 | Property Services LLC                    | -$2,225.00 | RENT
12/03/2024 | Premises Services                        | -$1,200.00 | NOT RENT
06/08